In [67]:
import numpy as np
import time
import os
import ase
from pyscf import gto, dft, df, lib
from pyscf.scf import hf
import scipy
from equiv_dens.utils import base as utils
%cd /home/mihail/Documents/workspace/equiv_dens/
hf.MUTE_CHKFILE = True
%load_ext autoreload
%autoreload 2

/home/mihail/Documents/workspace/equiv_dens


In [ ]:
# mol = gto.M(atom='O  0  0  0.1184; H  0,  0.7532, -0.4735; H 0,  -0.7532, -0.4735 ', basis='def2svp')
# mf = dft.RKS(mol)
# mf.chkfile=False
# mf.xc = 'pbe'
# mf.kernel()
# g = mf.nuc_grad_method()
# g.kernel()
data = np.load('datasets/h2o_dynamic_centered.npy', allow_pickle=True).item()
basis = 'augccpvdz'
auxbasis = 'augccpvqzjkfit'

atom_types = data['atom_types']

print(len(data['positions']))
save_path = 'datasets/h2o_dynamic_augccpvdz_df_augccpvqzjkfit.npy'
npy_path = 'datasets/h2o_dynamic_augccpvdz.npy'
if os.path.exists(save_path):
    results = list(np.load(save_path, allow_pickle=True))
else:
    results = []
print('results len', len(results))
for i in range(len(results), len(data['positions'])):
    print('calc', i)
    start = time.time()
    pos = data['positions'][i]
    atom = []
    for j in range(len(atom_types)):
        atom.append((atom_types[j], pos[j, :])) 
    mol = gto.M(atom=atom, basis=basis)
    #print(mol.pack())
    mf = dft.RKS(mol)
    mf.chkfile=False
    mf.xc = 'pbe'
    mf.kernel()
    g = mf.nuc_grad_method()
    gradients = g.grad()
    print('elapsed', time.time() - start)
    #print(mfs[i].mo_coeff)
    res = []
    res.append(mol.pack())
    calc_dict = {}
    calc_dict['mo_coeff'] = mf.mo_coeff
    calc_dict['mo_occ'] = mf.mo_occ
    calc_dict['energy'] = mf.e_tot
    calc_dict['forces'] = -gradients/ase.units.Bohr

    dm1 = mf.make_rdm1(mf.mo_coeff, mf.mo_occ)
    auxmol = df.addons.make_auxmol(mol, auxbasis)

    ints_3c2e = df.incore.aux_e2(mol, auxmol, intor='int3c2e')
    ints_2c2e = auxmol.intor('int2c2e')
    print('ints3c2e shape', ints_3c2e.shape)
    print('ints2c2e shape', ints_2c2e.shape)

    nao = mol.nao
    naux = auxmol.nao
    df_coef = scipy.linalg.solve(ints_2c2e, ints_3c2e.reshape(nao*nao, naux).T)
    df_coef = df_coef.reshape(naux, nao, nao)
    if dm1.ndim > 2:
        df_basis = []
        for j in range(dm1.shape[0]):
            df_basis.append(lib.einsum('Pij,ij->P', df_coef, dm1[j]))
        df_basis = np.stack(df_basis, axis=0)
        print(df_basis.shape)

    else:
        df_basis = lib.einsum('Pij,ij->P', df_coef, dm1)

    calc_dict['df_coeff'] = df_basis
    calc_dict['auxbasis'] = auxbasis
    res.append(calc_dict)
    results.append(res)

    if i%100 == 0:
        np.save(save_path, results, allow_pickle=True)
np.save(save_path, results, allow_pickle=True)
npy_data = utils.calc_dict_to_npy(results, convert_forces=False, compress_atoms=False)
np.save(npy_path, npy_data, allow_pickle=True)

In [81]:
# calculating for a single molecule, and getting different energy components
data = np.load('datasets/h2o_dynamic_centered.npy', allow_pickle=True).item()
basis = 'augccpvdz'

atom_types = data['atom_types']

i = 0
print('calc', i)
start = time.time()
pos = data['positions'][i]
atom = []
for j in range(len(atom_types)):
    atom.append((atom_types[j], pos[j, :]))
mol = gto.M(atom=atom, basis=basis)
#print(mol.pack())
mf = hf.RHF(mol)
mf.max_cycle = 0
mf.init_guess = 'atom'
mf.chkfile=False
mf.kernel()
print(mf.energy_tot())
print(mf.e_tot)

calc 0
SCF not converged.
SCF energy = -75.9848985824873
-75.9518927809653
-75.98489858248735


In [58]:
dm = mf.make_rdm1()
m_kin = mol.intor('int1e_kin')
m_nuc = mol.intor('int1e_nuc')

e_kin = np.einsum('ij,ji', dm, m_kin)
e_nuc = np.einsum('ij,ji', dm, m_nuc)

veff = mf.get_veff()
ecoul = veff.ecoul
exc = veff.exc

print('total energy', e_kin + e_nuc + ecoul + exc + mf.energy_nuc())
print(mf.__dir__())
print('total_energy etot', mf.e_tot)
print('total_energy', mf.energy_tot())
print('nuclear energy', mf.energy_nuc())
print('eletronic energy, coulomb energy', mf.energy_elec())
print(mf.get_veff().shape)
print(mf.mo_coeff.shape)

total energy -76.32730819782378
['mol', 'verbose', 'max_memory', 'stdout', 'chkfile', 'mo_energy', 'mo_coeff', 'mo_occ', 'e_tot', 'converged', 'callback', 'scf_summary', 'opt', '_eri', '_keys', 'xc', 'nlc', 'grids', 'nlcgrids', 'small_rho_cutoff', '_numint', 'init_guess', '_t0', '_w0', '__module__', '__doc__', '__init__', 'dump_flags', 'get_veff', 'get_vsap', 'energy_elec', 'init_guess_by_vsap', 'nuc_grad_method', 'Gradients', 'TDA', 'TDHF', 'TDDFTNoHybrid', 'CasidaTDDFT', 'TDDFT', 'dTDA', 'dRPA', 'omega', 'define_xc_', 'to_rhf', 'to_uhf', 'to_ghf', 'to_hf', 'to_rks', 'to_uks', 'to_gks', 'reset', 'initialize_grids', '__dict__', '__weakref__', '__new__', '__repr__', '__hash__', '__str__', '__getattribute__', '__setattr__', '__delattr__', '__lt__', '__le__', '__eq__', '__ne__', '__gt__', '__ge__', '__reduce_ex__', '__reduce__', '__subclasshook__', '__init_subclass__', '__format__', '__sizeof__', '__dir__', '__class__', 'check_sanity', 'get_jk', 'convert_from_', 'spin_square', 'stability'

In [59]:
def get_energy_components(mol, mf):
    """
    Get energy components for a single molecule.

    Args:
        mol: pyscf molecule
        mf: pyscf scf object
    Returns:
        energies: dictionary of energy components
    """
    dm = mf.make_rdm1()
    m_kin = mol.intor('int1e_kin')
    m_nuc = mol.intor('int1e_nuc')
    h1e = mf.get_hcore()
    veff = mf.get_veff()

    energies = {}
    energies['energy'] = mf.energy_tot()
    energies['energy_e_kin'] = np.einsum('ij,ji', dm, m_kin)
    energies['energy_e_nuc'] = np.einsum('ij,ji', dm, m_nuc)
    energies['energy_coul'] = veff.ecoul
    energies['energy_exc'] = veff.exc
    energies['energy_nuc'] = mf.energy_nuc()
    # print('energies', energies)
    # print('total energy', energies['energy'])
    # print('mf energy elec', mf.energy_elec())
    # print('mf energy nuc', mf.energy_nuc())
    # print('mf energy elec + nuc', mf.energy_elec() + mf.energy_nuc())
    # print('mf ecoul', energies['energy_coul'] + energies['energy_exc'])
    # print('energy h1e', energies['energy_e_kin'] + energies['energy_e_nuc'])
    # print('mf h1e', np.einsum('ij,ji', dm, h1e))
    #
    # print('total elec', energies['energy_e_kin'] + energies['energy_e_nuc'] +
    #       energies['energy_coul'] + energies['energy_exc'])
    # print('mf elec', np.einsum('ij,ji', dm, h1e) + energies['energy_coul'] + energies['energy_exc'])
    # print('summed components', energies['energy_e_kin'] + energies['energy_e_nuc'] +
    #       energies['energy_coul'] + energies['energy_exc'] + energies['energy_nuc'])

    assert np.isclose(energies['energy'], energies['energy_e_kin'] + energies['energy_e_nuc'] +
                      energies['energy_coul'] + energies['energy_exc'] + energies['energy_nuc'])
    return energies

In [69]:
set_types = ['train', 'valid', 'test']
for set_type in set_types:
    data = np.load('datasets/h2o_small_' + set_type + '_augccpvdz.npy', allow_pickle=True).item()
    basis = 'augccpvdz'
    auxbasis = 'augccpvqzjkfit'

    print(len(data['positions']))
    save_path = 'datasets/h2o_small_' + set_type + '_dft_augccpvdz_energy_comps_calc.npy'
    npy_path = 'datasets/h2o_small_' + set_type + '_dft_augccpvdz_energy_comps.npy'
    if os.path.exists(save_path):
        results = list(np.load(save_path, allow_pickle=True))
    else:
        results = []
    print('results len', len(results))
    for i in range(len(results), len(data['positions'])):
        print('calc', i)
        start = time.time()
        print('data positions shape', data['positions'].shape)
        pos = data['positions'][i]
        anums = data['atom_numbers'][i]
        print('pos shape', pos.shape)
        atom = []
        for j in range(len(anums)):
            atom.append((anums[j], pos[j, :])) 
        mol = gto.M(atom=atom, basis=basis)
        res = []
        res.append(mol.pack())
        #print(mol.pack())
        mf = dft.RKS(mol)
        mf.init_guess = 'atom'
        mf.max_cycle = 0
        mf.chkfile=False
        mf.xc = 'pbe'
        mf.kernel()
        g = mf.nuc_grad_method()
        gradients = g.grad()
        energies_SAD = get_energy_components(mol, mf)
        energies_SAD = {k + '_SAD': v for k, v in energies_SAD.items()}
        calc_dict = {}
        calc_dict.update(energies_SAD)
        calc_dict['forces_SAD'] = -gradients/ase.units.Bohr
        mol = gto.M(atom=atom, basis=basis)
        #print(mol.pack())
        mf = dft.RKS(mol)
        mf.chkfile=False
        mf.xc = 'pbe'
        mf.kernel()
        g = mf.nuc_grad_method()
        gradients = g.grad()
        energies = get_energy_components(mol, mf)

        calc_dict['forces'] = -gradients/ase.units.Bohr
        calc_dict.update(energies)

        print('calc_dict', calc_dict)
        res.append(calc_dict)
        results.append(res)

        if i%10 == 0:
            np.save(save_path, results, allow_pickle=True)
    np.save(save_path, results, allow_pickle=True)
    npy_data = utils.calc_dict_to_npy(results, convert_forces=False, compress_atoms=False)
    npy_data_compressed = utils.calc_dict_to_npy(results, convert_forces=False, compress_atoms=True)
    print('atom_number nc', npy_data['atom_numbers'][:3])
    print('atom_number c', npy_data_compressed['atom_numbers'][:3])
    print('pos nc', npy_data['positions'][:3])
    print('pos c', npy_data_compressed['positions'][:3])
    print('forces nc', npy_data['forces'][:3])
    print('forces c', npy_data_compressed['forces'][:3])
    print('forces sad nc', npy_data['forces_SAD'][:3])
    print('forces sad c', npy_data_compressed['forces_SAD'][:3])
    np.save(npy_path, npy_data, allow_pickle=True)

100
results len 100
energy_SAD
key has energy or forces
energy_e_kin_SAD
key has energy or forces
energy_e_nuc_SAD
key has energy or forces
energy_coul_SAD
key has energy or forces
energy_exc_SAD
key has energy or forces
energy_nuc_SAD
key has energy or forces
forces_SAD
key has energy or forces
forces
key has energy or forces
energy
key has energy or forces
energy_e_kin
key has energy or forces
energy_e_nuc
key has energy or forces
energy_coul
key has energy or forces
energy_exc
key has energy or forces
energy_nuc
key has energy or forces
energy_SAD
key has energy or forces
energy_e_kin_SAD
key has energy or forces
energy_e_nuc_SAD
key has energy or forces
energy_coul_SAD
key has energy or forces
energy_exc_SAD
key has energy or forces
energy_nuc_SAD
key has energy or forces
forces_SAD
key has energy or forces
forces
key has energy or forces
energy
key has energy or forces
energy_e_kin
key has energy or forces
energy_e_nuc
key has energy or forces
energy_coul
key has energy or forces


converged SCF energy = -76.3144711530811
--------------- RKS gradients ---------------
         x                y                z
0 H    -0.0064633256     0.0082220480    -0.0366792574
1 H     0.0272405443    -0.0161746192     0.1091000050
2 O    -0.0207766632     0.0079505700    -0.0724125304
----------------------------------------------
calc_dict {'energy_SAD': -76.09601471606662, 'energy_e_kin_SAD': 78.4686506867251, 'energy_e_nuc_SAD': -202.4324506970225, 'energy_coul_SAD': 49.52515767152303, 'energy_exc_SAD': -9.579171833794504, 'energy_nuc_SAD': 7.921799456501925, 'forces_SAD': array([[ 0.04195285,  0.07518387, -0.07838356],
       [-0.04981179,  0.00618663, -0.14191757],
       [ 0.00785762, -0.08136636,  0.22028557]]), 'forces': array([[ 0.01221392, -0.01553742,  0.06931375],
       [-0.05147717,  0.0305656 , -0.20616913],
       [ 0.0392622 , -0.0150244 ,  0.13683985]]), 'energy': -76.31447115308114, 'energy_e_kin': 75.53859742137537, 'energy_e_nuc': -196.14054180692435, 'e

SCF not converged.
SCF energy = -76.3180789259779
--------------- RKS gradients ---------------
         x                y                z
0 H     0.0170961451    -0.0062634549    -0.0021355293
1 H    -0.0015088312    -0.0080658157     0.1463706697
2 O    -0.0155852759     0.0143276533    -0.1442316096
----------------------------------------------
converged SCF energy = -76.3466378051518
--------------- RKS gradients ---------------
         x                y                z
0 H    -0.0404530546     0.0122915720     0.0479358320
1 H     0.0194501368    -0.0098604848     0.0439561252
2 O     0.0210051194    -0.0024331349    -0.0918875658
----------------------------------------------
calc_dict {'energy_SAD': -76.15071743117515, 'energy_e_kin_SAD': 78.77743122469163, 'energy_e_nuc_SAD': -204.3784901378636, 'energy_coul_SAD': 50.18363791651074, 'energy_exc_SAD': -9.662026584750212, 'energy_nuc_SAD': 8.928730150236492, 'forces_SAD': array([[-0.03230703,  0.01183621,  0.00403557],
    

calc_dict {'energy_SAD': -76.15009892511529, 'energy_e_kin_SAD': 79.00355130409599, 'energy_e_nuc_SAD': -205.58494093101007, 'energy_coul_SAD': 50.536810640753664, 'energy_exc_SAD': -9.710445384031166, 'energy_nuc_SAD': 9.604925445076706, 'forces_SAD': array([[ 0.31242734,  0.17803573,  0.04054905],
       [ 0.07557508, -0.24668352,  0.04656579],
       [-0.38799928,  0.0686459 , -0.08711133]]), 'forces': array([[ 0.14027309,  0.14543513,  0.00989551],
       [-0.02763719, -0.11773916,  0.00935084],
       [-0.11263491, -0.02769824, -0.0192437 ]]), 'energy': -76.33994275004531, 'energy_e_kin': 76.11955480425475, 'energy_e_nuc': -199.67140172529724, 'energy_coul': 46.87600325458663, 'energy_exc': -9.269024528665927, 'energy_nuc': 9.604925445076706}
calc 46
data positions shape (100, 3, 3)
pos shape (3, 3)
SCF not converged.
SCF energy = -76.3078532184755
--------------- RKS gradients ---------------
         x                y                z
0 H     0.0286934226     0.0175062510    -0

SCF not converged.
SCF energy = -76.2512047224874
--------------- RKS gradients ---------------
         x                y                z
0 H     0.0108423287    -0.0284263748    -0.0022734710
1 H     0.0058309055     0.0014900896    -0.0280060592
2 O    -0.0166721788     0.0269346106     0.0302780710
----------------------------------------------
converged SCF energy = -76.3345383150153
--------------- RKS gradients ---------------
         x                y                z
0 H     0.0252064201    -0.0364401393    -0.0526149645
1 H     0.0232877045    -0.0521630404    -0.0190780047
2 O    -0.0484932423     0.0886012225     0.0716913831
----------------------------------------------
calc_dict {'energy_SAD': -76.13584418709328, 'energy_e_kin_SAD': 78.35104223908642, 'energy_e_nuc_SAD': -202.46356001119923, 'energy_coul_SAD': 49.34545498799005, 'energy_exc_SAD': -9.561937259972696, 'energy_nuc_SAD': 8.193155857001797, 'forces_SAD': array([[-0.02048903,  0.05371806,  0.00429624],
   

...

SCF energy = -76.3272063928922
--------------- RKS gradients ---------------
         x                y                z
0 H    -0.0264766130    -0.0525354514    -0.0070796110
1 H    -0.0299181774     0.0641448704    -0.0663837650
2 O     0.0563937704    -0.0116161544     0.0734612379
----------------------------------------------
converged SCF energy = -76.3566884162655
--------------- RKS gradients ---------------
         x                y                z
0 H     0.0168452012     0.0102195587     0.0154733721
1 H     0.0038508343     0.0165932230    -0.0032020094
2 O    -0.0206963314    -0.0268190425    -0.0122724064
----------------------------------------------
calc_dict {'energy_SAD': -76.16873240710785, 'energy_e_kin_SAD': 78.67881585170736, 'energy_e_nuc_SAD': -204.1626484755496, 'energy_coul_SAD': 50.00465462499209, 'energy_exc_SAD': -9.643686400980982, 'energy_nuc_SAD': 8.954131992723314, 'forces_SAD': array([[ 0.05003355,  0.09927762,  0.01337853],
       [ 0.05653716, -0

calc_dict {'energy_SAD': -76.16303635020424, 'energy_e_kin_SAD': 78.67564504426888, 'energy_e_nuc_SAD': -204.14125496460053, 'energy_coul_SAD': 49.98750143797756, 'energy_exc_SAD': -9.64165974491913, 'energy_nuc_SAD': 8.956731877069245, 'forces_SAD': array([[ 0.00656831,  0.09321071,  0.02121588],
       [ 0.16955196, -0.12100324,  0.02657464],
       [-0.17611555,  0.02779244, -0.04779082]]), 'forces': array([[-0.05839386, -0.0252393 , -0.02295027],
       [ 0.02301502, -0.06044215, -0.00546977],
       [ 0.03538386,  0.08568204,  0.0284189 ]]), 'energy': -76.35219447842766, 'energy_e_kin': 75.8730248353609, 'energy_e_nuc': -198.3035832521418, 'energy_coul': 46.32776933322332, 'energy_exc': -9.206137271939058, 'energy_nuc': 8.956731877069245}
calc 83
data positions shape (100, 3, 3)
pos shape (3, 3)
SCF not converged.
SCF energy = -76.3142598429128
--------------- RKS gradients ---------------
         x                y                z
0 H    -0.0634575986    -0.1134753241    -0.001

SCF not converged.
SCF energy = -76.3486756724469
--------------- RKS gradients ---------------
         x                y                z
0 H     0.0897189283    -0.0449222623    -0.0084085853
1 H    -0.0558557889    -0.0329651083     0.1053519652
2 O    -0.0338577212     0.0778843537    -0.0969412866
----------------------------------------------
converged SCF energy = -76.3553653833638
--------------- RKS gradients ---------------
         x                y                z
0 H     0.0044780845    -0.0100628752     0.0124312082
1 H    -0.0009586152    -0.0154788197     0.0263106628
2 O    -0.0035137399     0.0255384107    -0.0387393336
----------------------------------------------
calc_dict {'energy_SAD': -76.16488622271238, 'energy_e_kin_SAD': 78.87495615318853, 'energy_e_nuc_SAD': -204.95104570876686, 'energy_coul_SAD': 50.36771644566388, 'energy_exc_SAD': -9.686579407782832, 'energy_nuc_SAD': 9.230066294984718, 'forces_SAD': array([[-0.1695442 ,  0.08489077,  0.01588992],
   

calc_dict {'energy_SAD': -76.1419943672264, 'energy_e_kin_SAD': 78.56955439366506, 'energy_e_nuc_SAD': -203.65676673630045, 'energy_coul_SAD': 49.77536027660797, 'energy_exc_SAD': -9.615264909937599, 'energy_nuc_SAD': 8.78512260873881, 'forces_SAD': array([[ 0.07840397,  0.08050919, -0.01009519],
       [-0.11524151, -0.07832223, -0.0564639 ],
       [ 0.03684294, -0.00219015,  0.06656997]]), 'forces': array([[ 0.05485279,  0.00592158,  0.08275505],
       [-0.04112683, -0.06734773,  0.05005248],
       [-0.01371979,  0.06142508, -0.13279819]]), 'energy': -76.3349495762879, 'energy_e_kin': 75.82158430630692, 'energy_e_nuc': -197.88386146172945, 'energy_coul': 46.12673888475273, 'energy_exc': -9.184533914356887, 'energy_nuc': 8.78512260873881}
calc 94
data positions shape (100, 3, 3)
pos shape (3, 3)
SCF not converged.
SCF energy = -76.2387710114209
--------------- RKS gradients ---------------
         x                y                z
0 H    -0.0030099512     0.0071545204     0.0010

SCF not converged.
SCF energy = -76.368823328116
--------------- RKS gradients ---------------
         x                y                z
0 H    -0.0471827635     0.0778141171     0.0131836556
1 H     0.2080819352    -0.0782844172    -0.1517986637
2 O    -0.1609041660     0.0004683063     0.1386157349
----------------------------------------------
converged SCF energy = -76.3198314821138
--------------- RKS gradients ---------------
         x                y                z
0 H     0.0239109356     0.0041698769    -0.0220980480
1 H     0.1094990325    -0.0164008039    -0.0886479716
2 O    -0.1334139888     0.0122291392     0.1107460801
----------------------------------------------
calc_dict {'energy_SAD': -76.11373876232477, 'energy_e_kin_SAD': 79.20740221921834, 'energy_e_nuc_SAD': -206.08665207379488, 'energy_coul_SAD': 50.923668088900115, 'energy_exc_SAD': -9.748574146800536, 'energy_nuc_SAD': 9.590417150151984, 'forces_SAD': array([[ 0.0891625 , -0.14704737, -0.0249135 ],
   

In [ ]:
np.save(save_path, results, allow_pickle=True)

In [ ]:
results = []
for i in range(len(mfs)):
    #print(mfs[i].mo_coeff)
    mol_dict = mols[i].pack()
    calc_dict = {}
    calc_dict['mo_coeff'] = mfs[i].mo_coeff
    calc_dict['mo_occ'] = mfs[i].mo_occ
    calc_dict['energy'] = mfs[i].e_tot
    calc_dict['forces'] = forces[i]
    results.append((mol_dict, calc_dict))
    results.append(res)

np.save('datasets/h2o_dynamic_pyscf_631gss_dft_f.npy', results)

In [ ]:
data_scf = np.load('datasets/h2o_dynamic_pyscf_631gss_dft_f.npy', allow_pickle=True)
print(data_scf)
print(len(data_scf))

In [ ]:
data_scf = np.load('datasets/h2o_dynamic_pyscf_dft_f.npy', allow_pickle=True)
print(data_scf)
print(len(data_scf))
for d in data_scf:
    print('energy', d['energy'])
    print('forces', d['forces'])

In [ ]:
for d in data_scf:
    d['forces'] = d['forces'] * 0.529177


np.save('datasets/h2o_dynamic_pyscf_dft_f.npy', data_scf)

In [ ]:
new_data = []
data_scf = data_scf = np.load('datasets/h2o_dynamic_pyscf_dft_f.npy', allow_pickle=True)
for d in data_scf:
    new_d = []
    mo_coeff = d.pop('mo_coeff')
    mo_occ = d.pop('mo_occ')
    en = d.pop('energy')
    f = d.pop('forces')
    new_d.append(d)
    new_d.append({'mo_coeff': mo_coeff, 'mo_occ': mo_occ, 'energy': en, 'forces': f})
    new_data.append(new_d)

np.save('datasets/h2o_dynamic_pyscf_dft_f_en.npy', new_data)

In [ ]:
results = {'E': [], 'F': [], 'R': [], 'z': np.array([8, 1, 1])}
for i in range(len(mfs)):
    #print(mfs[i].mo_coeff)
    res = mols[i].pack()
    pos = []
    for a in res['atom']:
        pos.append(a[1])
    pos = np.array(pos)
    print('pos.shape', pos.shape)
    results['R'].append(pos)
    results['E'].append(mfs[i].e_tot)
    results['F'].append(forces[i])
    
results['R'] = np.array(results['R'])
results['E'] = np.array(results['E'])
results['F'] = np.array(results['F'])

np.savez('datasets/water_pyscf_dft_f', **results)

In [ ]:
results = {'energy': [], 'forces': [], 'positions': [],
           'atom_numbers': [8, 1, 1], 'atom_types': ['O', 'H', 'H'],
          'mo_coeff': [], 'mo_occ': []}
for d in data_scf:
    #print(mfs[i].mo_coeff)
    pos = []
    for a in d['atom']:
        pos.append(a[1])
    pos = np.array(pos)
    print('pos.shape', pos.shape)
    results['positions'].append(pos)
    results['energies'].append(d['energy'])
    results['forces'].append(d['forces'])
    results['mo_coeff'].append(d['mo_coeff'])
    results['mo_occ'].append(d['mo_occ'])

for key in results.keys():
    results[key] = np.array(results[key])
    
np.savez('datasets/h2o_dynamic_pyscf_dft_f', **results)

In [ ]:
print(np.load('datasets/h2o_dynamic_pyscf_dft.npy', allow_pickle=True)[0])
print(np.load('datasets/h2o_dynamic_pyscf_dft_f.npy', allow_pickle=True)[0])
print(np.load('datasets/h2o_dynamic_pyscf_dft_f_en.npy', allow_pickle=True)[0])